# MyKepatuhan — RAG Evaluation

**Approach:** LlamaIndex built-in evaluators.


**Metrics:**
- `FaithfulnessEvaluator` — is the answer grounded in retrieved context?
- `RelevancyEvaluator` — is the answer relevant to the question?
- `CorrectnessEvaluator` — is the answer correct vs the reference answer?
- Latency — per-component timing

## 1 — Install

In [1]:

%pip install llama-index-llms-ollama

print('dependencies ready')

Note: you may need to restart the kernel to use updated packages.
dependencies ready


## 2 — Imports

In [17]:
import os, sys, time, json, warnings
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from llama_index.llms.ollama import Ollama


warnings.filterwarnings('ignore')
load_dotenv()

from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    CorrectnessEvaluator,
)

GEMINI_KEY = os.getenv('GEMINI_KEY')

judge_llm = Ollama(
    model='gemma4:e4b',
    base_url='http://localhost:11434',
    request_timeout=180,
)

faithfulness_eval  = FaithfulnessEvaluator(llm=judge_llm)
relevancy_eval     = RelevancyEvaluator(llm=judge_llm)
correctness_eval   = CorrectnessEvaluator(llm=judge_llm)

print('Evaluators: Faithfulness, Relevancy, Correctness    Ready!')

Evaluators: Faithfulness, Relevancy, Correctness    Ready!


## 3 — Load RAG Pipeline

In [18]:
sys.path.append('..')

from backend.pipeline.retriever import build_query_engine, build_retriever

query_engine = build_query_engine()
retriever    = build_retriever()

print('RAG pipeline loaded')

RAG pipeline loaded


## 4 — Load Test Questions

In [19]:
from questions import TEST_QUESTIONS

print(f'Total questions: {len(TEST_QUESTIONS)}')
print(f'  English:      {sum(1 for q in TEST_QUESTIONS if q["lang"] == "en")}')
print(f'  Bahasa Melayu: {sum(1 for q in TEST_QUESTIONS if q["lang"] == "bm")}')

by_topic = {}
for q in TEST_QUESTIONS:
    by_topic[q['topic']] = by_topic.get(q['topic'], 0) + 1
print(f'  By topic: {by_topic}')

Total questions: 5
  English:      5
  Bahasa Melayu: 0
  By topic: {'registration': 5}


## 5 — Latency Benchmark

In [20]:
import nest_asyncio
nest_asyncio.apply()

BENCHMARK_Q = 'To which geographical area does the Registration of Businesses Act 1956 apply?'
latency = {}

# Retrieval only
t0 = time.perf_counter()
nodes = retriever.retrieve(BENCHMARK_Q)
latency['retrieval_ms'] = round((time.perf_counter() - t0) * 1000, 1)

# Full pipeline (retrieval + rerank + LLM)
t0 = time.perf_counter()
response = query_engine.query(BENCHMARK_Q)
latency['end_to_end_ms'] = round((time.perf_counter() - t0) * 1000, 1)
latency['llm_ms'] = round(latency['end_to_end_ms'] - latency['retrieval_ms'], 1)

print(f'Retrieval:     {latency["retrieval_ms"]} ms ({len(nodes)} nodes)')
print(f'LLM + rerank:  {latency["llm_ms"]} ms')
print(f'End-to-end:    {latency["end_to_end_ms"]} ms')
print(f'\nAnswer preview: {response.response[:200]}...')

Retrieval:     4510.5 ms (15 nodes)
LLM + rerank:  3274.4 ms
End-to-end:    7784.9 ms

Answer preview: The Registration of Businesses Act 1956 applies to Peninsular Malaysia and the Federal Territory of Labuan....


## 6 — Run RAG on All Questions

In [21]:
rag_outputs = []
latency_records = []

print(f'Running {len(TEST_QUESTIONS)} questions through RAG pipeline...\n')

for i, item in enumerate(TEST_QUESTIONS):
    t0 = time.perf_counter()
    try:
        resp = query_engine.query(item['question'])
        elapsed = (time.perf_counter() - t0) * 1000

        contexts = [
            node.node.text
            for node in resp.source_nodes
            if node.node.text.strip()
        ]

        if not contexts:
            print(f'  [{i+1:02d}]  No contexts — skipping ({item["lang"].upper()})')
            continue

        rag_outputs.append({
            'question':  item['question'],
            'answer':    resp.response,
            'contexts':  contexts,
            'reference': item['reference'],
            'lang':      item['lang'],
            'topic':     item['topic'],
            'authority': item['authority'],
        })
        latency_records.append({
            'question':   item['question'][:55] + '...',
            'lang':       item['lang'],
            'topic':      item['topic'],
            'latency_ms': round(elapsed, 1),
        })
        print(f'  [{i+1:02d}] {item["lang"].upper()} | {item["topic"]:<12} | {elapsed:.0f} ms | {len(contexts)} ctx')

    except Exception as e:
        print(f'  [{i+1:02d}] ERROR: {e}')

df_latency = pd.DataFrame(latency_records)
print(f'\n Collected {len(rag_outputs)}/{len(TEST_QUESTIONS)} outputs')
print(f'   Mean latency: {df_latency["latency_ms"].mean():.0f} ms')
print(f'   P95 latency:  {df_latency["latency_ms"].quantile(0.95):.0f} ms')

Running 5 questions through RAG pipeline...

  [01] EN | registration | 6653 ms | 3 ctx
  [02] EN | registration | 7107 ms | 3 ctx
  [03] EN | registration | 2714 ms | 3 ctx
  [04] EN | registration | 2229 ms | 3 ctx
  [05] EN | registration | 1705 ms | 3 ctx

 Collected 5/5 outputs
   Mean latency: 4082 ms
   P95 latency:  7016 ms


## 7 — Evaluate 

In [7]:
import asyncio
from llama_index.core.schema import TextNode
from llama_index.core.base.response.schema import Response

async def run_evaluation():
    records = []

    print(f'Evaluating {len(rag_outputs)} outputs with Gemini judge...\n')
    print('(Each question makes 3 Gemini calls — takes ~2-3 min total)\n')

    for i, row in enumerate(rag_outputs):
        print(f'  [{i+1:02d}/{len(rag_outputs)}] {row["question"][:60]}...')

        source_nodes = [TextNode(text=ctx) for ctx in row['contexts']]
        response_obj = Response(
            response=row['answer'],
            source_nodes=source_nodes,
        )

        record = {
            'question':   row['question'],
            'answer':     row['answer'][:200],
            'lang':       row['lang'],
            'topic':      row['topic'],
            'authority':  row['authority'],
            'n_contexts': len(row['contexts']),
        }

        # ── Faithfulness ──────────────────────────────────────────
        try:
            r = await faithfulness_eval.aevaluate_response(
                query=row['question'],
                response=response_obj,
            )
            record['faithfulness']          = r.score
            record['faithfulness_pass']     = r.passing
            record['faithfulness_feedback'] = r.feedback
            print(f'       faithfulness:  {r.score} | pass={r.passing}')
        except Exception as e:
            record['faithfulness']          = None
            record['faithfulness_pass']     = None
            record['faithfulness_feedback'] = None
            print(f'       faithfulness error: {e}')

        # ── Relevancy ─────────────────────────────────────────────
        try:
            r = await relevancy_eval.aevaluate_response(
                query=row['question'],
                response=response_obj,
            )
            record['relevancy']          = r.score
            record['relevancy_pass']     = r.passing
            record['relevancy_feedback'] = r.feedback
            print(f'       relevancy:     {r.score} | pass={r.passing}')
        except Exception as e:
            record['relevancy']          = None
            record['relevancy_pass']     = None
            record['relevancy_feedback'] = None
            print(f'       relevancy error: {e}')

        # ── Correctness ───────────────────────────────────────────
        try:
            r = await correctness_eval.aevaluate_response(
                query=row['question'],
                response=response_obj,
                reference=row['reference'],
            )
            record['correctness']          = r.score
            record['correctness_pass']     = r.passing
            record['correctness_feedback'] = r.feedback
            print(f'       correctness:   {r.score} | pass={r.passing}')
        except Exception as e:
            record['correctness']          = None
            record['correctness_pass']     = None
            record['correctness_feedback'] = None
            print(f'       correctness error: {e}')

        records.append(record)
        await asyncio.sleep(1)  

    return records

# Run it
eval_records = asyncio.get_event_loop().run_until_complete(run_evaluation())
df_eval = pd.DataFrame(eval_records)
print(f'\n✅ Evaluation complete — {len(df_eval)} results')
print(df_eval[['lang', 'topic', 'faithfulness', 'relevancy', 'correctness']].to_string())

Evaluating 5 outputs with Gemini judge...

(Each question makes 3 Gemini calls — takes ~2-3 min total)

  [01/5] To which geographical area does the Registration of Business...
       faithfulness:  1.0 | pass=True
       relevancy:     0.0 | pass=False
       correctness:   3.0 | pass=False
  [02/5] Within what timeframe must the person responsible for a busi...
       faithfulness:  1.0 | pass=True
       relevancy:     1.0 | pass=True
       correctness:   4.0 | pass=True
  [03/5] What is the maximum period for which the Registrar can regis...
       faithfulness:  1.0 | pass=True
       relevancy:     1.0 | pass=True
       correctness:   5.0 | pass=True
  [04/5] Where must the certificate of registration issued for a main...
       faithfulness:  1.0 | pass=True
       relevancy:     1.0 | pass=True
       correctness:   5.0 | pass=True
  [05/5] What is the status of the Minister's decision regarding an a...
       faithfulness:  1.0 | pass=True
       relevancy:     1.0 | pass=Tr

## 8 — Results

In [11]:
SCORE_COLS = ['faithfulness', 'relevancy', 'correctness']

# ── Overall ──────────────────────────────────────────────────────────
print('=' * 55)
print('OVERALL SCORES')
print('(faithfulness & relevancy: 0-1 | correctness: 1-5)')
print('=' * 55)
for col in SCORE_COLS:
    if col in df_eval.columns:
        mean = df_eval[col].mean()
        passing = df_eval.get(f'{col}_pass', pd.Series()).mean()
        print(f'  {col:<16} mean={mean:.3f}   pass rate={passing:.0%}')

# ── By language ───────────────────────────────────────────────────────
print('\n' + '=' * 55)
print('BY LANGUAGE')
print('=' * 55)
print(df_eval.groupby('lang')[SCORE_COLS].mean().round(3).to_string())

# ── By topic ──────────────────────────────────────────────────────────
print('\n' + '=' * 55)
print('BY TOPIC')
print('=' * 55)
print(df_eval.groupby('topic')[SCORE_COLS].mean().round(3).to_string())

# ── By authority ──────────────────────────────────────────────────────
print('\n' + '=' * 55)
print('BY AUTHORITY')
print('=' * 55)
print(df_eval.groupby('authority')[SCORE_COLS].mean().round(3).to_string())

# ── Latency ───────────────────────────────────────────────────────────
print('\n' + '=' * 55)
print('LATENCY')
print('=' * 55)
print(f'  Mean:   {df_latency["latency_ms"].mean():.0f} ms')
print(f'  Median: {df_latency["latency_ms"].median():.0f} ms')
print(f'  P95:    {df_latency["latency_ms"].quantile(0.95):.0f} ms')

OVERALL SCORES
(faithfulness & relevancy: 0-1 | correctness: 1-5)
  faithfulness     mean=1.000   pass rate=100%
  relevancy        mean=0.800   pass rate=80%
  correctness      mean=4.200   pass rate=80%

BY LANGUAGE
      faithfulness  relevancy  correctness
lang                                      
en             1.0        0.8          4.2

BY TOPIC
              faithfulness  relevancy  correctness
topic                                             
registration           1.0        0.8          4.2

BY AUTHORITY
                                  faithfulness  relevancy  correctness
authority                                                             
Companies Commission of Malaysia           1.0        0.8          4.2

LATENCY
  Mean:   4683 ms
  Median: 2968 ms
  P95:    10389 ms


## 9 — Weakest Answers (what to improve)

In [14]:
for metric in ['faithfulness', 'correctness']:
    if metric not in df_eval.columns:
        continue
    print(f'\n── Lowest {metric} ──────────────────────────────')
    worst = df_eval.nsmallest(3, metric)[['question', metric, f'{metric}_feedback', 'lang']]
    for _, row in worst.iterrows():
        print(f'  Score: {row[metric]:.2f} | {row["lang"].upper()}')
        print(f'  Q: {row["question"][:80]}')
        feedback = str(row.get(f'{metric}_feedback', ''))[:500]
        print(f'  Feedback: {feedback}')
        print()


── Lowest faithfulness ──────────────────────────────
  Score: 1.00 | EN
  Q: To which geographical area does the Registration of Businesses Act 1956 apply?
  Feedback: YES

  Score: 1.00 | EN
  Q: Within what timeframe must the person responsible for a business apply for regis
  Feedback: YES

  Score: 1.00 | EN
  Q: What is the maximum period for which the Registrar can register a business under
  Feedback: YES


── Lowest correctness ──────────────────────────────
  Score: 3.00 | EN
  Q: To which geographical area does the Registration of Businesses Act 1956 apply?
  Feedback: The generated answer is relevant because it attempts to answer the geographical scope of the Act. However, it differs from the reference answer ("Peninsular Malaysia only") by including "the Federal Territory of Labuan." Without further context, this addition suggests a potential factual error or incomplete scope, placing the answer in the "relevant but contains mistakes" category.

  Score: 4.00 | EN
  Q: Wi

## 10 — Save Results

In [15]:
ts = datetime.now().strftime('%Y%m%d_%H%M')

df_eval.to_csv(f'results/eval_{ts}.csv', index=False)
df_latency.to_csv(f'results/latency_{ts}.csv', index=False)

summary = {
    'timestamp': ts,
    'n_questions': len(df_eval),
    'scores': {col: round(df_eval[col].mean(), 4) for col in SCORE_COLS if col in df_eval.columns},
    'pass_rates': {col: round(df_eval[f'{col}_pass'].mean(), 4) for col in SCORE_COLS if f'{col}_pass' in df_eval.columns},
    'latency': {
        'mean_ms':   round(df_latency['latency_ms'].mean(), 1),
        'median_ms': round(df_latency['latency_ms'].median(), 1),
        'p95_ms':    round(df_latency['latency_ms'].quantile(0.95), 1),
    }
}

with open(f'results/summary_{ts}.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved to evaluation/results/')
print(json.dumps(summary, indent=2))

✅ Saved to evaluation/results/
{
  "timestamp": "20260524_1619",
  "n_questions": 5,
  "scores": {
    "faithfulness": 1.0,
    "relevancy": 0.8,
    "correctness": 4.2
  },
  "pass_rates": {
    "faithfulness": 1.0,
    "relevancy": 0.8,
    "correctness": 0.8
  },
  "latency": {
    "mean_ms": 4683.2,
    "median_ms": 2967.7,
    "p95_ms": 10388.6
  }
}
